# Dataset para modelo

In [13]:
from pathlib import Path
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

ROOT = Path().resolve().parents[0]
DATA = Path(ROOT/"data")
OUT = Path(ROOT/"data/processed")
OUT.mkdir(exist_ok=True)

In [14]:
src = pd.read_csv(DATA / "processed/reviews_eda.csv", encoding="utf-8-sig")
print(src.shape)

src.head(3)

(300, 26)


,id,rating,polaridad3,titulo,texto,texto_completo,n_caracteres,n_palabras,tipo_viaje,fecha_estadia,...,t_atencion,t_comida,t_habitaciones,t_piscina_playa,t_mantenimiento,t_limpieza,t_ubicacion,t_checkin_checkout,t_precio,temas
0,1050834500,1,negativa,Mala experiencia en hotel Dorado Cartagena,"Nos alojaron en habitaciones impresentables, t...",Mala experiencia en hotel Dorado Cartagena. No...,376,60,COUPLES,2026-02-01,...,1,0,1,0,0,1,0,0,0,Habitaciones|Limpieza e higiene|Atención y per...
1,1044265712,3,neutra,Vacaciones familiares,"Buena ubicación, necesita mantenimiento y mejo...","Vacaciones familiares. Buena ubicación, necesi...",278,49,FAMILY,2025-12-01,...,0,1,0,0,1,1,1,0,0,Comida y bebidas|Mantenimiento e instalaciones...
2,1043382081,1,negativa,Pésimo servicio al cliente,El hotel no cumple con la oferta de valor de a...,Pésimo servicio al cliente. El hotel no cumple...,565,103,FRIENDS,2025-12-01,...,1,0,0,0,1,0,1,1,0,Atención y personal|Mantenimiento e instalacio...


### 1.1 Qué columnas se conservan y cuáles se descartan
| Columna | Decisión | Rol | Motivo |
|---|---|---|---|
| `id` | mantenido | identificador | trazabilidad, permite verificar que no haya reseñas en ambos archivos, nunca se utiliza como variable de entrada. |
| `titulo`, `texto` | descartado | ninguno | se utiliza texto_completo. |
| `texto_completo` | mantenido | es la combinación de título+texto. |
| `rating` | mantenido | etiqueta | polaridad en 5 clases. |
| `polaridad3` | mantenido | etiqueta | negativa/neutra/positiva, derivada de rating. |
| `tema_principal` | mantenido | etiqueta | tema en 9 clases. |
| `t_*` | descartado | ninguno | son las etiquetas de temas en formato binario, redundantes. |
| `fecha_publicación`, `semana`, `mes` | descartado | ninguno | si el modelo ve la fecha, puede aprender cosas como 2025 es negativo, las fechas sirven para el simulador, no para el modelo,. |
| `tipo_viaje`, `fecha_estadia` | descartado | ninguno | sin relación útil con la polaridad, el agente no las usa. |
| `n_caracteres`, `n_palabras` | descartado | ninguno | derivadas del texto. |
| `temas` | mantenido | etiqueta secundaria | lista separada por `\|`, sirve para la métrica permisiva |
| `n_temas` | descartado | ninguno | sale de la etiqueta de temas |
| `hotel`, `lenguaje` | descartado | ninguno | son constantes (un solo hotel y un solo idioma). |

In [15]:
# Etiqueta para cada reseña, para que la división sea estratificada
def clave_estrato(d, min_celda=4):
    neg = pd.Series(np.where(d["polaridad3"] =="negativa", "neg", "no_neg"), index=d.index)
    clave = d["tema_principal"].astype(str).copy()
    # Combina polaridad con el tema principal para dividir las categorías
    for tema, idx in d.groupby("tema_principal").groups.items():
        if neg.loc[idx].value_counts().min() >= min_celda:
            # clave de estrato "tema|neg" o "tema|no_neg"
            clave.loc[idx] = tema + "|" + neg.loc[idx]
    return clave

# Columnas del modelo a mantener
COLS_MODELO = ["id", "texto_completo", "rating",
               "polaridad3", "tema_principal", "temas"]
base = src[COLS_MODELO].copy()

In [16]:
train, test = train_test_split(base, test_size=0.30, stratify=clave_estrato(src), random_state=42)

assert set(train["id"]).isdisjoint(
    test["id"]) and len(train) + len(test) == len(src)
train.to_csv(DATA/"processed/train.csv",
             index=False, encoding="utf-8-sig")
test.to_csv(ROOT/"data/processed/test.csv", index=False, encoding="utf-8-sig")

# Verifique que se parezcan
pd.DataFrame({"total": src["tema_principal"].value_counts(),
              "train": train["tema_principal"].value_counts(),
              "test": test["tema_principal"].value_counts()})

,total,train,test
tema_principal,,,
Atención y personal,123,86,37
Check-in / Check-out y reservas,12,9,3
Comida y bebidas,47,33,14
Habitaciones,54,38,16
Limpieza e higiene,8,6,2
Mantenimiento e instalaciones,29,20,9
"Piscina, playa y entretenimiento",13,9,4
Precio / relación calidad-precio,9,6,3
Ubicación,5,3,2
